In [2]:
import json

In [3]:
gsm_traces = json.load(open("/local-storage/interaction/siyanli/DP_PAPILLON/papillon/GSM8k_Qwen_Qwen2.5-1.5B-Instruct_2026-03-10_12:21:41.json"))

In [4]:
REP_STR = "<|im_start|>system\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\n<|im_start|>assistant"
# for i in range(23):
INSPECT_NUM = 10
cs = gsm_traces[INSPECT_NUM]["comps"]
cs = [x.replace(REP_STR, "") for x in cs]
print(INSPECT_NUM, "=======================")
for c in cs:
    print(c)
    print("---------")
    

10 =======================

Tobias saved for three months, so he saved 3 * $5 = $15 from his allowance. He also mowed 4 lawns, earning 4 * $15 = $60 from mowing lawns. He earned a total of $15 + $60 = $75 from his allowance and mowing lawns. He spent $95 on the shoes, so he had $75 - $95 = $-20 left after buying the shoes. Since he had $15 in change, he actually spent $-20 - $15 = $-35. He earned $75 from mowing lawns, so he must have shoveled 75 / 7 = 10.71 driveways. Since he can't shovel a fraction of a driveway, he shoveled 11 driveways. The answer is 11.<|im_end|>

---------

James has been saving for three months, so he saved 3 * $5 = $15 from his allowance. He also mowed 4 lawns, earning 4 * $15 = $60 from mowing lawns. He also shoveled driveways, earning 7 * number of driveways. After buying the shoes, he has $15 left, so the total amount he spent on the shoes is $95 - $15 = $80. The total amount he earned from mowing lawns and shoveling driveways is $60 + $80 = $140. Since he 

In [5]:
import re

REP_STR = "<|im_start|>system\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\n<|im_start|>assistant"

def extract_final_answer(completion: str):
    """Extract the number after 'The final answer is' from a completion string."""
    completion = completion.replace(",", "")
    match = re.findall(r"\d+", completion)
    if match:
        return match[-1].strip()
    return None

# For each group, extract answers and check if all variants agree
all_same, all_none, groups_with_disagreement = [], [], []

ifs_scores = json.load(open("/local-storage/interaction/siyanli/DP_PAPILLON/papillon/ifs_GSM8k_Qwen_Qwen2.5-1.5B-Instruct_2026-03-10_12:21:41.json.json"))
ifs_corr = []
num_corr = []

for i, group in enumerate(gsm_traces):
    comps = [c.replace(REP_STR, "") for c in group["comps"]]
    answers = [extract_final_answer(c) for c in comps]
    non_null = [a for a in answers if a is not None]

    if not non_null:
        all_none.append(i)
        continue

    unique_answers = set(non_null)
    if len(unique_answers) == 1 and len(non_null) == len(answers):
        all_same.append(i)
    else:
        groups_with_disagreement.append({
            "group": i,
            "answers": answers,
            "unique": unique_answers,
        })
    if str(i) in ifs_scores:
        ifs_corr.append(ifs_scores[str(i)]["ifs_normalized"])
        num_corr.append(len(unique_answers))


print(f"Total groups          : {len(gsm_traces)}")
print(f"All variants agree    : {len(all_same)} ({100*len(all_same)/len(gsm_traces):.1f}%)")
print(f"Disagreement          : {len(groups_with_disagreement)} ({100*len(groups_with_disagreement)/len(gsm_traces):.1f}%)")
print(f"No answer extracted   : {len(all_none)} ({100*len(all_none)/len(gsm_traces):.1f}%)")


Total groups          : 1180
All variants agree    : 709 (60.1%)
Disagreement          : 471 (39.9%)
No answer extracted   : 0 (0.0%)


In [6]:
# Inspect a few disagreement cases
for g in groups_with_disagreement[:5]:
    i = g["group"]
    print(f"Group {i} | unique answers: {g['unique']}")
    for variant_idx, (prompt, answer) in enumerate(zip(gsm_traces[i]["prompts"], g["answers"])):
        print(f"  variant {variant_idx}: answer={answer}")
    print()


Group 2 | unique answers: {'5', '55'}
  variant 0: answer=55
  variant 1: answer=5
  variant 2: answer=5
  variant 3: answer=5
  variant 4: answer=5
  variant 5: answer=5
  variant 6: answer=5
  variant 7: answer=5
  variant 8: answer=5
  variant 9: answer=5
  variant 10: answer=5
  variant 11: answer=5
  variant 12: answer=5
  variant 13: answer=5
  variant 14: answer=5
  variant 15: answer=5

Group 4 | unique answers: {'312', '624'}
  variant 0: answer=624
  variant 1: answer=624
  variant 2: answer=312
  variant 3: answer=312
  variant 4: answer=624
  variant 5: answer=312
  variant 6: answer=624
  variant 7: answer=624
  variant 8: answer=624
  variant 9: answer=624
  variant 10: answer=624
  variant 11: answer=624
  variant 12: answer=312
  variant 13: answer=624
  variant 14: answer=312
  variant 15: answer=624

Group 5 | unique answers: {'35', '22'}
  variant 0: answer=35
  variant 1: answer=22
  variant 2: answer=35
  variant 3: answer=35
  variant 4: answer=22
  variant 5: ans

In [7]:
from scipy.stats import spearmanr
spearmanr(ifs_corr, num_corr)

SignificanceResult(statistic=np.float64(0.331156223520091), pvalue=np.float64(3.899166385769096e-09))

In [8]:
from constants import MALE_NAME_LIST, FEMALE_NAME_LIST

name_cts, total_cts = 0, 0

for i, group in enumerate(gsm_traces):
    comps = [c.replace(REP_STR, "") for c in group["comps"]]
    for c in comps:
        for n in MALE_NAME_LIST + FEMALE_NAME_LIST:
            if n in c:
                name_cts += 1
                break
        total_cts += 1
    total_cts -= 1
print(name_cts / float(total_cts))

0.9140677966101695
